In [ ]:
# ==============================
# Carga de librerías y dataset
# ==============================
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import copy
from torch.utils.data import DataLoader, TensorDataset

# Cargamos el dataset preprocesado generado en datos.ipynb
# Contiene 22 features (retardos, volatilidades, retornos individuales) + target (L_{t+1})
dataset = pd.read_pickle("dataset_tfg.pkl").copy()
dataset = dataset.sort_index()
assert isinstance(dataset.index, pd.DatetimeIndex), "El índice debe ser DatetimeIndex"

# Nivel de confianza del VaR: el modelo estimará el cuantil 0.99 de L_{t+1}
alpha = 0.99

# Columnas de features (todas menos la variable objetivo)
feature_cols = [c for c in dataset.columns if c != "target"]

In [ ]:
def var_loss(y_true, y_pred, q=0.99, w=10.0):
    """
    Función de pérdida Pinball (Quantile Loss) ponderada para VaR.

    Parámetros:
        q : cuantil objetivo (0.99 → VaR al 99%)
        w : penalización extra aplicada cuando hay violación (y_true > y_pred)
            w=10.0 → las violaciones se penalizan 10× más que el conservadurismo

    A mayor w, el modelo aprende a ser más conservador (VaR más alto),
    reduciendo la tasa de violación a costa de un mayor capital inmovilizado.
    """
    e = y_true - y_pred

    # Si hay violación (pérdida real > VaR predicho), aplicamos peso w; si no, peso 1
    weight = torch.where(e > 0, w, 1.0)

    # Función pinball: max(q·e, (q-1)·e) — equivale a la pérdida cuantílica estándar
    pinball = torch.maximum(q * e, (q - 1) * e)

    return torch.mean(weight * pinball)

In [ ]:
class MLPVaR(nn.Module):
    """
    Red neuronal multicapa (MLP) para la estimación directa del VaR al 99%.

    Arquitectura:
        Entrada (22 features) → Linear(128) → ReLU → Linear(64) → ReLU → Linear(1)

    La salida es escalar: el cuantil predicho de L_{t+1} (VaR del día siguiente).
    El bias inicial de 0.02 da un arranque razonable (~2% de pérdida diaria),
    evitando gradientes nulos en las primeras épocas de entrenamiento.
    """
    def __init__(self, d_in, dropout=0.2):
        super().__init__()
        self.body = nn.Sequential(
            nn.Linear(d_in, 128),
            nn.ReLU(),
            #nn.Dropout(dropout),   # desactivado: la ventana rodante actúa como regularización
            nn.Linear(128, 64),
            nn.ReLU(),
            #nn.Dropout(dropout),
            nn.Linear(64, 1)
        )

        # Inicialización del bias de salida: valor inicial del VaR ≈ 2%
        nn.init.constant_(self.body[-1].bias, 0.02)

    def forward(self, x):
        return self.body(x)

In [ ]:
# ==============================
# Entrenamiento con Early Stopping
# ==============================

def train_one_model(X_train, y_train, d_in, seed,
                    alpha=0.99,
                    max_epochs=200,
                    lr=5e-5,
                    batch_size=256,
                    patience=15,
                    val_split=0.10):
    """
    Entrena una instancia del MLPVaR con early stopping sobre el conjunto de validación.

    Parámetros:
        X_train    : features normalizados (numpy float32, shape [N, 22])
        y_train    : pérdidas reales del día siguiente (numpy float32, shape [N, 1])
        seed       : semilla aleatoria para reproducibilidad
        max_epochs : número máximo de épocas de entrenamiento
        lr         : tasa de aprendizaje del optimizador Adam
        batch_size : tamaño de mini-batch para el DataLoader
        patience   : épocas consecutivas sin mejora antes de parar
        val_split  : fracción del conjunto usada como validación (temporal)

    Devuelve el modelo con los pesos del mejor epoch (menor pérdida de validación).
    """
    torch.manual_seed(seed)
    np.random.seed(seed)

    n = len(X_train)
    split = int(n * (1 - val_split))

    X_tr, X_val = X_train[:split], X_train[split:]
    y_tr, y_val = y_train[:split], y_train[split:]

    model = MLPVaR(d_in=d_in)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    train_loader = DataLoader(
        TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(y_tr)),
        batch_size=batch_size,
        shuffle=False
    )

    X_val_t = torch.from_numpy(X_val)
    y_val_t = torch.from_numpy(y_val)

    best_loss = np.inf
    best_state = None
    patience_counter = 0

    for epoch in range(max_epochs):

        # ---- TRAIN: pase hacia adelante + retropropagación ----
        model.train()
        for xb, yb in train_loader:
            opt.zero_grad()
            pred = model(xb)
            loss_val = var_loss(yb, pred, q=alpha)
            loss_val.backward()
            opt.step()

        # ---- VALIDACIÓN: evaluamos sin acumular gradientes ----
        model.eval()
        with torch.no_grad():
            val_pred = model(X_val_t)
            val_loss = var_loss(y_val_t, val_pred, q=alpha).item()

        # ---- EARLY STOP: guardamos el mejor estado del modelo ----
        if val_loss < best_loss - 1e-4:
            best_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model

In [ ]:
from tqdm import tqdm

# ==============================
# Bucle de evaluación con ventana deslizante (rolling window)
# ==============================
eval_start = pd.Timestamp("2015-01-01")
eval_end   = pd.Timestamp("2025-01-01")
eval_dates = dataset.loc[eval_start:eval_end].index
retrain_every = 1  # reentrenamiento diario

seed = 42  # semilla fija para reproducibilidad

var_preds = []
real_targets = []
dates = []
current_model = None
muX, sdX = None, None  # estadísticos de normalización del entrenamiento actual

print(f"\n{'='*40}")
print(f"Evaluando seed fija = {seed}")
print(f"{'='*40}")

for i, current_date in enumerate(
    tqdm(eval_dates, desc="Rolling NN VaR", dynamic_ncols=True)
):

    if i % retrain_every == 0:
        # Ventana deslizante: 900 obs anteriores al día actual (sin incluirlo)
        train_end = current_date - pd.Timedelta(days=1)
        train_df = dataset.loc[:train_end].tail(900)

        if len(train_df) < 250:
            if current_model is None:
                continue
        else:
            X_train = train_df[feature_cols].values.astype(np.float32)
            y_train = train_df["target"].values.astype(np.float32).reshape(-1,1)

            # Normalización z-score exclusiva del entrenamiento (sin data leakage)
            muX = X_train.mean(axis=0, keepdims=True)
            sdX = X_train.std(axis=0, keepdims=True) + 1e-8
            X_train = (X_train - muX) / sdX

            current_model = train_one_model(
                X_train, y_train,
                d_in=X_train.shape[1],
                seed=seed
            )

    if current_model is None or muX is None or sdX is None:
        continue

    # Predicción del VaR para el día actual con la normalización del entrenamiento
    test_df = dataset.loc[[current_date]]
    X_test = (test_df[feature_cols].values.astype(np.float32) - muX) / sdX
    y_test = test_df["target"].values.astype(np.float32).reshape(-1,1)

    current_model.eval()
    with torch.no_grad():
        pred = current_model(torch.from_numpy(X_test)).numpy().reshape(-1)[0]

    var_preds.append(pred)
    real_targets.append(y_test.reshape(-1)[0])
    dates.append(current_date)

In [ ]:
# ==============================
# Resultados finales
# ==============================
eval_df = pd.DataFrame(
    {"VaR_pred": var_preds, "loss_real": real_targets},
    index=pd.DatetimeIndex(dates)
).loc["2015":"2024"]

# Indicador de violación: 1 si la pérdida real supera el VaR estimado
eval_df["viol"] = (eval_df["loss_real"] > eval_df["VaR_pred"]).astype(int)
eval_df["year"] = eval_df.index.year

# Violation rate total e interanual
total_vr = eval_df["viol"].mean()
vr_year = eval_df.groupby("year")["viol"].mean()

print("\n" + "="*40)
print("RESULTADOS FINALES")
print("="*40)
print(f"Violation rate total (2015-2024): {total_vr:.4f}")

print("\nViolation rate por año:")
for year, vr in vr_year.items():
    print(f"{year}: {vr:.4f}")

In [ ]:
# Exportamos las predicciones para el análisis comparativo en comp_final.ipynb
eval_df.to_csv("nn_var_predictions_10.csv")